In [ ]:
# Cargar variables de entorno desde el archivo .env
import os
from pathlib import Path
from dotenv import load_dotenv

# Cargar .env desde la raíz del proyecto
env_path = Path('../.env')
if not env_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo .env en {env_path.absolute()}")

load_dotenv(dotenv_path=env_path)

# Leer variables requeridas (falla si no existen)
required_vars = ['MINIO_ACCESS_KEY', 'MINIO_SECRET_ACCESS_KEY', 'MINIO_PORT', 'MLFLOW_PORT']
missing_vars = [var for var in required_vars if not os.getenv(var)]

if missing_vars:
    raise EnvironmentError(f"Faltan las siguientes variables en .env: {', '.join(missing_vars)}")

# Configurar variables de entorno para MLflow y MinIO
os.environ['AWS_ACCESS_KEY_ID'] = os.getenv('MINIO_ACCESS_KEY')
os.environ['AWS_SECRET_ACCESS_KEY'] = os.getenv('MINIO_SECRET_ACCESS_KEY')

# Construir URLs dinámicamente desde .env
MINIO_PORT = os.getenv('MINIO_PORT')
MLFLOW_PORT = os.getenv('MLFLOW_PORT')

os.environ['MLFLOW_S3_ENDPOINT_URL'] = f'http://localhost:{MINIO_PORT}'
MLFLOW_TRACKING_URI = f'http://localhost:{MLFLOW_PORT}'

print("✓ Variables de entorno cargadas desde .env")
print(f"  AWS_ACCESS_KEY_ID: {os.environ['AWS_ACCESS_KEY_ID']}")
print(f"  MLFLOW_S3_ENDPOINT_URL: {os.environ['MLFLOW_S3_ENDPOINT_URL']}")
print(f"  MLFLOW_TRACKING_URI: {MLFLOW_TRACKING_URI}")

In [ ]:
# Verificar que las variables están configuradas
print(f"AWS_ACCESS_KEY_ID: {os.environ.get('AWS_ACCESS_KEY_ID')}")
print(f"MLFLOW_S3_ENDPOINT_URL: {os.environ.get('MLFLOW_S3_ENDPOINT_URL')}")

In [3]:
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor

# Usar la URI de MLflow desde la variable cargada del .env
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# set the experiment id
experiment_name = "test_experiment"
mlflow.set_experiment(experiment_name)

mlflow.autolog()
db = load_diabetes()

X_train, X_test, y_train, y_test = train_test_split(db.data, db.target)

# Create and train models.
rf = RandomForestRegressor(n_estimators=100, max_depth=6, max_features=3)
rf.fit(X_train, y_train)

# Use the model to make predictions on the test dataset.
predictions = rf.predict(X_test)


2026/07/19 00:13:46 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/07/19 00:13:46 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'cc7b9f2898e94262ad563ca19fe98942', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/07/19 00:13:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/19 00:14:00 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.cl

🏃 View run awesome-sloth-972 at: http://localhost:5001/#/experiments/1/runs/cc7b9f2898e94262ad563ca19fe98942
🧪 View experiment at: http://localhost:5001/#/experiments/1
